# 04 — The Composite Inconsistency Index

`CII = w1·SSG + w2·ASCS + w3·ISG`. The research contribution.
Feeds **thesis 4.6** and **5.4**, and sets up the ablation in 5.10.

Three ways a review can contradict itself, each on [0, 1]:

| Component | Measures |
|---|---|
| SSG | the text's sentiment against the stars given |
| ASCS | aspect opinions pointing against the stars |
| ISG | emotional emphasis against the stars |


## Kernel check


In [ ]:
import sys; print(sys.executable)


## Build the components

Training-set components are computed **out of fold** — the step-3 and step-4 models were
fitted on the whole training set, so their predictions on training rows are in-sample.
Tuning the weights on those would overfit. Takes about a minute.


In [ ]:
import pandas as pd, numpy as np
from IPython.display import Image, display
from trustlens import config, cii
res = cii.report_all()
feat = res['features']
res['fit']


## Two definitions, and why

The pre-specified design maps stars onto sentiment and intensity linearly:
`(r−3)/2` and `|r−3|/2`. That works for sentiment but fails badly for intensity.

Mean intensity in this corpus is **0.10**, not the 1.0 the linear form assumes at five
stars. So `|0.10 − 1.0|` scored almost every plain five-star review — `"Great."`,
`"සුපිරියක්"` — as *maximally inconsistent*, and ISG measured nothing but the absence
of emoji.

The centred variants measure each gap from what a review at that rating actually looks
like, fitted on training data. Both are reported; `isg_linear` and `ssg` keep the
pre-specified definitions so the comparison is in the thesis rather than hidden.


In [ ]:
pd.DataFrame(res['fit']['expected_per_star'])


## Separation of Suspicious


In [ ]:
res['evaluation']


Read this carefully. Review length alone reaches **0.841** and beats every CII variant.
That is the length confound documented in `GOLD_DATASET.md`: the Suspicious label is
close to a rule about review length, so a trivial feature dominates.

Centring helps SSG a lot — 0.431 (below chance) to 0.563. ISG stays at chance either
way, and the weight search drops it to zero. Two of its four markers barely fire in this
corpus: elongation appears 0.015 times per review, punctuation 0.059.


## The headline result

The overall number hides what matters. Split by review length:


In [ ]:
res['stratified']


In [ ]:
display(Image(filename=str(config.FIGURES/'11_cii_auc_by_length.png'), width=620))


**Length and CII are complementary, not competing.**

Below 20 characters, length carries all the signal and CII adds nothing. At 20 characters
and above, length collapses to chance — 0.508, CI [0.318, 0.700] — while CII reaches
**0.825, CI [0.699, 0.924]**. At 40+ characters CII reaches 0.936.

CII is the only signal that works where length fails, and that is the defensible claim.

**State the caveat plainly.** The longer strata hold 10 and 6 Suspicious reviews. The
intervals are tight because the separation is near-complete, not because the sample is
large. A replication with more long suspicious reviews would strengthen this considerably,
and its absence is a limitation, not a footnote.


In [ ]:
display(Image(filename=str(config.FIGURES/'12_cii_distribution.png'), width=520))


## Face validity — what high CII actually finds


In [ ]:
from trustlens import data
g = data.load_gold().set_index('review_id')
te = feat[feat.split=='test']
for _, r in te.nlargest(8, 'cii').iterrows():
    row = g.loc[r.review_id]
    print(f"CII {r.cii:.2f} | {int(r.star_rating)}* | {row.authenticity}")
    print(f"   {str(row.review_text)[:90]}\n")


## What this means for the ablation in step 6

Do not report CII against the Suspicious label alone — length wins that comparison and
the result would read as a failure. Report three things:

1. CII's contribution **over and above a length-only model**, on the full test set.
2. The length-stratified result above, which is where the contribution is real.
3. Predicted versus annotated aspects, which quantifies what aspect-model error costs.

| Output | Section |
|---|---|
| `14_cii_auc.csv` | 5.4 — experimental results |
| `15_cii_weight_grid.csv` | 4.6 — weight derivation |
| `17_cii_auc_by_length.csv`, figure `11` | **5.10 — results against research objectives** |
| figure `12` | 5.4 |
